# High-performance and parallel computing for AI - Multi-device MPI, MPI4JAX, and equinox tutorial

IMPORTANT
=========

* When using a GPU we need to set some environment variables (see below). If you get some weird error restart the kernel and rerun the code below.
* For these practicals and tutorials we will be using a different `conda environment`. When opening a notebook or a terminal make sure you are using the **hpc4ai Kernel**!!!


**Note:** As of 03/06/2026 `mpi4jax` version `0.9.0.post1` has an installation bug and fails when communicating between GPUs. While the developers fix this we are setting the environment variable `MPI4JAX_USE_CUDA_MPI` to `0`. It will make things work, but all memory will move through the CPU. In the future when they fix it you can check it is fixed by setting `MPI4JAX_USE_CUDA_MPI` to `1`. If it's not fixed it will crash with a segfault.

In [1]:
import os

# JAX-specific environment variables
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.249" # restrict GPU memory preallocation to a fourth of the total
os.environ["JAX_OPTIMIZATION_LEVEL"]="O1"
os.environ["JAX_NUM_CPU_DEVICES"] = "6" # JAX will see 6 CPUs (if available)
os.environ["MPI4JAX_USE_CUDA_MPI"] = "0" # temporary fix while the mpi4jax developers fix the installation script.

## On goliat we have FIVE GPUs so here we pick two of those at random
## so that we do not overload the system.
## The way we do it is by figuring out the GPU UUIDs and then setting
## The CUDA_VISIBLE_DEVICES environment variable.
## Note: this is useful for other libraries as well (e.g., Jax, PyTorch, TF) in multi-GPU servers.

# To get these UUIDs you need to run nvidia-smi -q on the command line
quadro_UUIDs = ["GPU-4efa947b-abbd-7c6e-84f5-61241d34bb4b",
                "GPU-5eb524b0-2b1b-fe98-e6ed-b8fb5185e993"]

L40s_UUIDs = ["GPU-7bba1f33-03d2-016b-d42e-ced83c3ac243",
              "GPU-179d068a-3bea-91d7-1a8c-7017f55d6298",
              "GPU-ae634859-dd49-de46-9182-195639405eaa"]

import random

a, b = random.sample(range(3), 2) # picks two numbers between 0 and 2 at random

# Picks an L40s and a Quadro GPU at random. The others will be invisible to JAX
# NOTE: this only works if the environment variable is set BEFORE JAX is first imported.
os.environ["CUDA_VISIBLE_DEVICES"] = L40s_UUIDs[a] + "," + L40s_UUIDs[b]

## JAX (or whatever other software) will only see these GPUs.

######################## NOW WE CAN IMPORT STUFF ##########################

import jax
import jax.numpy as jnp
import equinox as eqx
from jax.flatten_util import ravel_pytree

## MPI with GPUs in general

To enable the use of MPI with GPUs you need to install a CUDA-aware version of MPI (currently, only OpenMPI supports it) and set a few environment variables. We did this for you so you do not have to worry about it.

The MPI rules and framework are then the same. The only difference will happen under the hood: communication between GPUs will happen directly without passing through the CPUs.

Typically, it is easier to assign a GPU to each rank so that each rank controls 1 CPU and 1 GPU. However, this is not mandatory, you can do whatever you want.

Here is a short example using CuPy:

**Note:** Here we use the same workflow using `%%writefile` as in the MPI tutorials and practicals and the same `mempool` trick to limit the GPU memory as in Practical 1.

In [2]:
%%writefile temp.py

from mpi4py import MPI
import cupy as cp

# Make sure we do not overload the GPU memory, this is just for these tutorials
mempool = cp.get_default_memory_pool()
mempool.set_limit(size=1.5*1024**3)  # 1.5 GiB

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# On the CPU we can work in the same way
x = rank + 1
total = comm.allreduce(x, op=MPI.SUM)

expected = size * (size + 1) // 2
assert expected == total

# Get number of GPUs that are visible (it should be 2 due to the way we set environment variables at the top of this notebook)
ndev = cp.cuda.runtime.getDeviceCount()
assert ndev == 2

# Assign one GPU to each rank
# Note: If there are more ranks than GPUs here we will assign
#       the same GPU to multiple ranks.
device_id = rank % ndev
cp.cuda.Device(device_id).use()

# By default CuPy allocates arrays on the GPU
x = cp.array([rank+1], dtype=cp.int32)
print("Rank: %d" % rank, " Array allocated on device: ", x.device, flush=True)

# allreduce directly between the different GPUs.
# Note that it works in the same way as usual
total = comm.allreduce(x, op=MPI.SUM)

# Copy the result to the CPU to compare it (expected is a CPU variable)
total_host = cp.asnumpy(total) 
assert expected == total_host[0]

comm.barrier()
if rank == 0:
    print("All good!", flush=True)

Overwriting temp.py


In [3]:
!mpirun -np 2 python3 temp.py

Rank: 1  Array allocated on device:  <CUDA Device 1>
Rank: 0  Array allocated on device:  <CUDA Device 0>
All good!


<hr style="height:4px; background-color:black; border:none; border-color:black;">

## Plain MPI in JAX

Following the same strategy we can use MPI within JAX without any extra software. This still requires CuPy.

In [4]:
%%writefile temp.py

from mpi4py import MPI
import numpy as np
import jax
import jax.numpy as jnp

import cupy as cp

# Make sure we do not overload the GPU memory, this is just for these tutorials
mempool = cp.get_default_memory_pool()
mempool.set_limit(size=1.5*1024**3)  # 1.5 GiB

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

ndev = cp.cuda.runtime.getDeviceCount() # Get number of CuPy GPUs
assert len(gpus) == ndev # Make sure JAX and CuPy see the same GPUs

# NOTE: this is slightly tricky. In the first cell of the notebook
# we told JAX to look for 6 CPUs. JAX does not care that we are also
# using MPI so each MPI rank will see 6 CPUs. We do not know if those will
# be the same or not. For this example this does not matter, but
# if you want to use sharding and MPI at the same time one has to be careful
# to set environment variables correctly.
assert len(cpus) == 6 and len(gpus) == 2

# Assign a GPU to each rank (and make sure that JAX and CuPy use the same GPUs)
device_id = rank % ndev
cp.cuda.Device(device_id).use()
device = gpus[device_id]

# Assign a CPU to each rank
cpu_id = rank % 6
host = cpus[cpu_id]

# Allocate an array on the GPU directly using JAX
x_jax = jnp.array([rank + 1], dtype=jnp.int32, device=device)

# For debugging, print device info
print(f"Rank {rank} JAX array device: {x_jax.device}", flush=True)

# Convert JAX array -> CuPy array without moving memory outside of the GPU
# Note: only works if the GPU used is the same for both JAX and CuPy
x_cupy = cp.from_dlpack(x_jax)

# allreduce directly between the different GPUs.
# Note that it works in the same way as usual
total_cupy = comm.allreduce(x_cupy, op=MPI.SUM)

# This is not needed, but we can cast the result into a new JAX array
# which we can then move to the host with device_put
# (note that we could have also directly used CuPy here)
total_jax = jax.dlpack.from_dlpack(total_cupy)
total_host = jax.device_put(total_jax, device=host)

# Most likely not needed, just making 100% sure expected lives on the same CPU
expected = jnp.array([size * (size + 1) // 2], dtype=jnp.int32, device=host)

assert expected[0] == total_host[0]

comm.barrier()
if rank == 0:
    print("All good!", flush=True)

Overwriting temp.py


In [5]:
!mpirun -np 2 python3 temp.py

Rank 1 JAX array device: cuda:1
Rank 0 JAX array device: cuda:0
All good!


<hr style="height:4px; background-color:black; border:none; border-color:black;">

## MPI4JAX

By using [mpi4jax](https://mpi4jax.readthedocs.io/en/latest/) things are much easier:

In [2]:
%%writefile temp.py

from mpi4py import MPI
import numpy as np
import jax
import jax.numpy as jnp

import mpi4jax

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6 and len(gpus) == 2

# Assign a GPU and CPU to each rank
device_id = rank % len(gpus)
device = gpus[device_id]
cpu_id = rank % len(cpus)
host = cpus[cpu_id]

# Allocate an array on the GPU directly using JAX
x = jnp.array([rank + 1], dtype=jnp.int32, device=device)

# For debugging, print device info
print(f"Rank {rank} JAX array device: {x.device}", flush=True)

# allreduce directly between the different GPUs.
# Note that it works in the same way as usual
# Except for three things: 1) allreduce is a mpi4jax member function.
#                          2) We need to pass the MPI communicator.
total = mpi4jax.allreduce(x, op=MPI.SUM, comm=comm)

expected = jnp.array([size * (size + 1) // 2], dtype=jnp.int32, device=device)

assert expected[0] == total[0]

# NOTE: DO NOT MIX mpi4jax and mpi4py calls in the same script!
mpi4jax.barrier(comm=comm)
if rank == 0:
    print("All good!", flush=True)

Overwriting temp.py


In [3]:
!mpirun -np 2 python3 temp.py

Rank 0 JAX array device: cuda:0
Rank 1 JAX array device: cuda:1
All good!


### MPI4JAX is JIT-compatible

In [8]:
%%writefile temp.py

from mpi4py import MPI
import numpy as np
import jax
import jax.numpy as jnp

import mpi4jax

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6 and len(gpus) == 2

# Assign a GPU and CPU to each rank
device_id = rank % len(gpus)
device = gpus[device_id]
cpu_id = rank % len(cpus)
host = cpus[cpu_id]

# Allocate an array on the GPU directly using JAX
x = jnp.array([rank + 1], dtype=jnp.int32, device=device)

# For debugging, print device info
print(f"Rank {rank} JAX array device: {x.device}", flush=True)

@jax.jit
def test(x):
    total = mpi4jax.allreduce(x, op=MPI.SUM, comm=comm)
    return total

expected = jnp.array([size * (size + 1) // 2], dtype=jnp.int32, device=device)

total = test(x)

assert expected[0] == total[0]

# NOTE: DO NOT MIX mpi4jax and mpi4py calls in the same script!
mpi4jax.barrier(comm=comm)
if rank == 0:
    print("All good!", flush=True)

Overwriting temp.py


In [9]:
!mpirun -np 2 python3 temp.py

Rank 0 JAX array device: cuda:0
Rank 1 JAX array device: cuda:1
All good!


<hr style="height:4px; background-color:black; border:none; border-color:black;">

## JAX Pytrees

Very quickly, just to show you how to work with pytress in JAX.

In [10]:
a = [[1, 2, 3], [1, 2], [1, 2, 3, 4]]

print("\nOriginal list of lists:\n", a)

b = jax.tree.map(lambda x: x*2, a) # multiplies by 2 all leaves

print("\nb = a^2:\n", b)

c = jax.tree.map(lambda x,y : x*y, a, b)

print("\nc = a*b:\n", c)

flat, unflatten = ravel_pytree(a)
# unflatten is a function that can be used to rebuild the original pytree
b = unflatten(flat) # b and a will be the same

print("\nFlattened a:\n", flat)
print("\nUnflattened a (Note that each leaf will be a jax.numpy array):\n", b)


Original list of lists:
 [[1, 2, 3], [1, 2], [1, 2, 3, 4]]

b = a^2:
 [[2, 4, 6], [2, 4], [2, 4, 6, 8]]

c = a*b:
 [[2, 8, 18], [2, 8], [2, 8, 18, 32]]

Flattened a:
 [1 2 3 1 2 1 2 3 4]

Unflattened a (Note that each leaf will be a jax.numpy array):
 [[Array(1, dtype=int32, weak_type=True), Array(2, dtype=int32, weak_type=True), Array(3, dtype=int32, weak_type=True)], [Array(1, dtype=int32, weak_type=True), Array(2, dtype=int32, weak_type=True)], [Array(1, dtype=int32, weak_type=True), Array(2, dtype=int32, weak_type=True), Array(3, dtype=int32, weak_type=True), Array(4, dtype=int32, weak_type=True)]]


<hr style="height:4px; background-color:black; border:none; border-color:black;">

## JAX Pytrees in EQUINOX

The JAX library `equinox` implement neural networks in a JAX native way by using pytrees.

All neural network architectures built with `equinox` are formed by a pytree and a static part:

* The **pytree part** contains all neural network parameters and possibly some additional array/container/pytree parameters.
* The **static part** contains everything else that cannot be put into a pytree, e.g., random number generator keys, functions, etc.

#### Manipulation of pytrees in equinox

To split a neural network stored into a variable `model` into its pytree and static parts use `eqx.partition`:

```python
params, static = eqx.partition(model, eqx.is_array)
```

`params` will then be a pytree containing the pytree part of the neural network model (typically its parameters, namely the biases and weights). `static` will contain the static part.

We can then use pytree manipulation functions, e.g., `jax.tree.map` or `ravel_pytree` to manipulate `params`. To reconstruct the neural network model we use `eqx.combine`:

```python
    model = eqx.combine(params, static)
```

Equinox also offers automatic ways of doing this via some helper decorators `@eqx.filter_grad`, `@eqx.filter_value_and_grad`, `@eqx.filter_jit`. These will automatically consider the static part as a static constant, which means the static part won't be differentiated and is taken as a known fixed constant by JIT. The use of these decorators is the same as the respective jax decorators. 

Here is now an example of an equinox MLP and how it can be easily modified by combining equinox helper functions and pytree manipulations.

In [11]:
# A 2-layer multilayer perceptron from equinox with tanh activation functions
model = eqx.nn.MLP(
    in_size=1,
    out_size=1,
    width_size=64,
    depth=2,
    activation=jax.nn.tanh,
    key=jax.random.PRNGKey(0),
)
print("\n", "#"*10, "\n\nThe original MLP:\n")

print(model)


 ########## 

The original MLP:

MLP(
  layers=(
    Linear(
      weight=f32[64,1],
      bias=f32[64],
      in_features=1,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[64,64],
      bias=f32[64],
      in_features=64,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[1,64],
      bias=f32[1],
      in_features=64,
      out_features=1,
      use_bias=True
    )
  ),
  activation=<PjitFunction of <function tanh at 0x7fcf36f2a5c0>>,
  final_activation=<function <lambda>>,
  use_bias=True,
  use_final_bias=True,
  in_size=1,
  out_size=1,
  width_size=64,
  depth=2
)


In [12]:
# Equinox-specific function: it separates the MLP into pytree (the NN parameters) + everything else that is not an array (e.g., the PRNGKey)
params, static = eqx.partition(model, eqx.is_array)

print("\n", "#"*10, "\n\nThe MLP as a pytree:\n")
print(params)
print("\n", "#"*10, "\n\nThe MLP without the parameters:\n")
print(static)


 ########## 

The MLP as a pytree:

MLP(
  layers=(
    Linear(
      weight=f32[64,1],
      bias=f32[64],
      in_features=1,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[64,64],
      bias=f32[64],
      in_features=64,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[1,64],
      bias=f32[1],
      in_features=64,
      out_features=1,
      use_bias=True
    )
  ),
  activation=None,
  final_activation=None,
  use_bias=True,
  use_final_bias=True,
  in_size=1,
  out_size=1,
  width_size=64,
  depth=2
)

 ########## 

The MLP without the parameters:

MLP(
  layers=(
    Linear(
      weight=None, bias=None, in_features=1, out_features=64, use_bias=True
    ),
    Linear(
      weight=None, bias=None, in_features=64, out_features=64, use_bias=True
    ),
    Linear(
      weight=None, bias=None, in_features=64, out_features=1, use_bias=True
    )
  ),
  activation=<PjitFunction of <function tanh at 0x7fcf36f2a5c0>>,
 

In [13]:
# Now params is a pytree and we can ravel it
mlp_parameters, unflatten_params = ravel_pytree(params)

print("\n", "#"*10, "\n\nThe flattened array containing all NN parameters:\n")
print(mlp_parameters)


 ########## 

The flattened array containing all NN parameters:

[-0.43456006  0.5761945  -0.20483303 ... -0.03326973  0.11982819
  0.06142175]


In [14]:
# To reconstruct the MLP we need to first unflatten the parameters into a pytree,
# and then use the equinox-specific function combine to reunite the pytree
# with the other variables which we had separated with partition
def unflatten(flat_params):
    params = unflatten_params(flat_params)          # pytree of arrays
    return eqx.combine(params, static)              # full model

# Example: round-trip check
model_reconstructed = unflatten(mlp_parameters)

print("\n", "#"*10, "\n\nThe reconstructed MLP:\n")
print(model_reconstructed)

# You can also use the following auxiliary function for flattening/unflattening
def flatten(model):
    params, static = eqx.partition(model, eqx.is_array)
    mlp_parameters, unflatten_params = ravel_pytree(params)
    def unflatten(flat_params):
        params = unflatten_params(flat_params)          # pytree of arrays
        return eqx.combine(params, static)

    return mlp_parameters, unflatten


 ########## 

The reconstructed MLP:

MLP(
  layers=(
    Linear(
      weight=f32[64,1],
      bias=f32[64],
      in_features=1,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[64,64],
      bias=f32[64],
      in_features=64,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[1,64],
      bias=f32[1],
      in_features=64,
      out_features=1,
      use_bias=True
    )
  ),
  activation=<PjitFunction of <function tanh at 0x7fcf36f2a5c0>>,
  final_activation=<function <lambda>>,
  use_bias=True,
  use_final_bias=True,
  in_size=1,
  out_size=1,
  width_size=64,
  depth=2
)


In [15]:
# In equinox gradients are also pytrees
# See for instance the following example

# Define a loss function
def loss_fn(model, x):
    return jnp.sum((model(x)-1)**2)

# The "filter" functions automatically use eqx.partition and eqx.combine
# so that we do not have to worry about it when computing gradients and JIT
# See the equinox documentation for further details
grad_fn = eqx.filter_jit(eqx.filter_grad(loss_fn))

x = jnp.ones((1,), dtype=jnp.float32)
model_grad = grad_fn(model, x)

print("\n", "#"*10, "\n\nThe gradient of the loss function is an equinox model!\n")
print(model_grad)

grad_flat,unflatten_grad = flatten(model_grad)
print("\n", "#"*10, "\n\nThe flattened gradient of the loss function (the way we normally think about a gradient):\n")
print(grad_flat)


 ########## 

The gradient of the loss function is an equinox model!

MLP(
  layers=(
    Linear(
      weight=f32[64,1],
      bias=f32[64],
      in_features=1,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[64,64],
      bias=f32[64],
      in_features=64,
      out_features=64,
      use_bias=True
    ),
    Linear(
      weight=f32[1,64],
      bias=f32[1],
      in_features=64,
      out_features=1,
      use_bias=True
    )
  ),
  activation=None,
  final_activation=None,
  use_bias=True,
  use_final_bias=True,
  in_size=1,
  out_size=1,
  width_size=64,
  depth=2
)

 ########## 

The flattened gradient of the loss function (the way we normally think about a gradient):

[ 0.02046983  0.01180722 -0.00755837 ...  0.0594344  -0.60288525
 -1.4999995 ]


In [16]:
# We can now modify the gradient as we want by using pytree manipulation functions
# For instance, we can do a gradient descent step!
# We now show three different strategies.

# Strategy 1) Use eqx.apply_update to add a pytree to a model
# Inputs: model + model gradient
update = jax.tree.map(lambda g : - 1.0e-3*g, model_grad)
gd_step_1 = eqx.apply_updates(model, update)

# Strategy 2) - Using jax.tree.map
# Inputs: pytree of model + pytree of model gradient
grad, _ = eqx.partition(model_grad, eqx.is_array)
params, static = eqx.partition(model, eqx.is_array)
new_model_params = jax.tree.map(lambda x,g : x - 1.0e-3*g, params, grad) # output is a pytree
gd_step_2 = eqx.combine(new_model_params, static)

# Strategy 3) - flattening both pytrees and modifying the flattened version
# Inputs: flattened pytree of model + flattened pytree of model gradient
model_flat, unflatten_model = flatten(model)
grad_flat,unflatten_grad = flatten(model_grad)
gd_step_flat = model_flat - 1.0e-3*grad_flat
gd_step_3 = unflatten_model(gd_step_flat)

# All these are equivalent. Strategy 1) is, in general, the most convenient.
# However, Strategy 3) works well with mpi4jax since we need to flatten the model pytrees for MPI communication.

assert jnp.allclose(flatten(gd_step_1)[0], flatten(gd_step_2)[0])
assert jnp.allclose(flatten(gd_step_2)[0], flatten(gd_step_3)[0])

print("All GD steps are equal!!!")

All GD steps are equal!!!
